# CLIP Clustering

This notebook walks you through the code that was experimented with to cluster the Glashelder dataset using CLIP embeddings.

The main technologies being used are:
- [CLIP pretrained model](https://huggingface.co/laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K)
- [UMAP dimensionality reduction algorithm](https://umap-learn.readthedocs.io/en/latest/)
- [HDBSCAN density based spacial clustering algorithm for applications with noise](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)

CLIP generates embeddings in 768 dimensions, which is too high for any clustering algorithm to process efficiently. \
Therefore, UMAP is used to partially reduce the dimensionality to 30 dimensions. \
The lower the dimensionality in which the clustering takes place, the faster, but going too low may 'squish' multiple distinct spaces onto eachother.

For visualizing the clusters, a second UMAP dimensionality reduction is used, to get the data visualizable in 3D.

The data flow of the code in this notebook looks like this:

```mermaid
flowchart LR
  i[images]@{shape: procs} -->|N images| c[CLIP]
  c -->|embedding 768xN| u1["UMAP(1)"]
  u1 -->|reduced 30xN| h
  u1 -->|reduced 30xN| u2["UMAP(2)"]
  u2 -->|reduced 3xN| v
  h[HDBSCAN] --> clusters@{shape: procs}
  h --> noise
  noise --> v
  clusters --> v
```



# Setup

### Dependencies

Import all necessary python libraries by running the cell below:

In [1]:
import glob

from hdbscan import HDBSCAN
from umap import UMAP

from cluster.rr import log_clustered_points, log_images_3d
from common import PROJECT_ROOT
from embed import CLIPEmbedder, get_device

### Configurations
- `MODEL`: the CLIP model to get from huggingface
- `PACKAGE_ROOT`: filesystem path of the directory with files related to the `cluster` experiment
- `EMBEDDINGS_PATH`: the location where the embeddings will be saved to in a file
- `IMAGE_DIR`: the path where you store your Glashelder images. **Modify the path below to where your Glashelder images are located**.

In [ ]:
MODEL = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"
PACKAGE_ROOT = PROJECT_ROOT / "cluster"
EMBEDDINGS_PATH = PACKAGE_ROOT / "embeddings.pkl"
IMAGE_DIR = "/mnt/share/shares/glashelder/assets-crop"  # MODIFY ME

Download the model and set it up for your device (cuda, cpu, mps):

In [4]:
device = get_device()
embedder = CLIPEmbedder(MODEL)
embedder.load(device)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/workspace/.venv/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.13/site-packages/transformers/safetensors_conv

## Make image embeddings

A random selection of images is chosen from within the image directory (defined above in `ÌMAGE_DIR`) that was specified before. The amount of images is limited here to a dataset of 4000 for speed. If you want a larger dataset to look through, increase the variable, `count`.

In [9]:
# this makes a 'random' (images have random order) selection of images in the dataset of limited size for speed

count = 4000 #If you want to select a larger portion of your dataset to look through, increase this variable
image_paths = sorted(glob.glob(f"{IMAGE_DIR}/*.jpg"))[:count]

Generate the embeddings of the images:

In [ ]:
embeddings, image_paths = embedder.encode_images(image_paths, batch_size=64)
embeddings.shape  # count vectors of length 768

torch.Size([4000, 768])

# Reduce embeddings' dimensionality

In the previous block of code we generated embeddings with 768 dimentions. This is too much to further process the images in a useful way. Therefore, we reduce them to 30 dimensions in the code block below. This makes handeling them easier:

In [7]:
# 30 dimensions for clustering
umap_model_30 = UMAP(n_components=30, n_neighbors=15, min_dist=0.20)
embeddings_30 = umap_model_30.fit_transform(embeddings)

While working with 30 dimentions is more efficient, for us humans it is impossible to visualize these. Therefore we redeuce the dimensions to three, so we can propperly see the clusters in a 3D environment in the later steps:

In [8]:
# 3 dimensions for visualization
umap_model_3 = UMAP(n_components=3, n_neighbors=15, min_dist=0.20)
embeddings_3 = umap_model_30.fit_transform(embeddings_30)

# Cluster in 30 dimensions

**Kort uitleggen wat deze twee blokjes code doen**

In [ ]:
clusterer = HDBSCAN(min_cluster_size=10)
clusterer.fit(embeddings_30)

In [ ]:
clusterer.labels_

# Show clusters
Chances are high that the clustering doesn't return desired results when you supply it a limited number of images. You can mitigate this by increasing the `count` variable at the top half of this notebook.

Import the necessary python libraries by running the cell below:

In [ ]:
import rerun as rr

**uitleg toevoegen over wat de code block hieronder doet**

In [ ]:
rr.init("cluster_notebook")
rr.save("clusters.rrd")
log_clustered_points(embeddings_3, clusterer.labels_, count=count)
log_images_3d(embeddings_3, image_paths, 100)

Open the saved file 'clusters.rrd' in the rerun viewer (e.g. by first downloading it from the jupyter UI)